In [ ]:
import pandas as pd
import numpy as np
import re
import zipfile
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- Start of Fix ---
# Check if the archive.zip file exists. If not, prompt the user to upload it.
if not os.path.exists('archive.zip'):
    print("archive.zip not found. Please upload the file.")
    from google.colab import files
    uploaded = files.upload()
    if 'archive.zip' not in uploaded:
        raise FileNotFoundError("archive.zip was not uploaded. Please upload the file to proceed.")
# --- End of Fix ---

# Unzip the archive
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('archive')

# Load datasets
fake = pd.read_csv("archive/Fake.csv")
real = pd.read_csv("archive/True.csv")

# Add labels
fake["label"] = 0
real["label"] = 1

# Combine datasets
data = pd.concat([fake, real])

# Shuffle dataset
data = data.sample(frac=1).reset_index(drop=True)

# Use only text and label columns
data = data[["text","label"]]


# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub("[^a-zA-Z]", " ", text)
    text = re.sub("\\s+", " ", text)
    return text

# Apply cleaning
data["text"] = data["text"].apply(clean_text)


# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.7)

X = vectorizer.fit_transform(data["text"])
y = data["label"]


# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# Train model
model = LogisticRegression()
model.fit(X_train, y_train)


# Predictions
pred = model.predict(X_test)


# Accuracy
accuracy = accuracy_score(y_test, pred)
print("Accuracy:", accuracy)


# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, pred))


# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, pred))


# Function to test custom news
def predict_news(news):

    news = clean_text(news)
    news_vector = vectorizer.transform([news])

    prediction = model.predict(news_vector)

    if prediction[0] == 1:
        print("Real News")
    else:
        print("Fake News")


# Test the model
print("\nTest with your own news\n")

user_news = input("Enter news text: ")

predict_news(user_news)


Accuracy: 0.9858574610244989

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4756
           1       0.98      0.99      0.98      4224

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980


Confusion Matrix:

[[4687   69]
 [  58 4166]]

Test with your own news



In [4]:
import pickle

# save model
pickle.dump(model, open("fake_news_model.pkl", "wb"))

# save vectorizer
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))